# Phase 7 — Evaluation & Comparison

**Goal:** Run all 7 methods on all 4 datasets with unified metrics and produce the final comparison figures for the report.

| Metric | What it measures | Best value |
|---|---|---|
| **Trustworthiness** | False neighbours in embedding (tears) | 1.0 |
| **Continuity** | Missing neighbours from original space (compressions) | 1.0 |
| **Spearman r** | Rank correlation of pairwise distances | 1.0 |
| **Recon MSE** | Autoencoder reconstruction error (AE only) | 0.0 |

**Methods compared:** PCA · Isomap · LLE · LLE Modified · Spectral Embedding · UMAP · Autoencoder

**Kernel:** `manifold-discovery`

In [ ]:
# Cell 1 — Imports & path fix
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd())
while not (ROOT / 'environment.yml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'ROOT = {ROOT}  |  src found: {(ROOT / "src").exists()}')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import pandas as pd
import torch

from src.datasets.generators import load_all_datasets
from src.methods.pca_isomap  import run_pca, run_isomap
from src.methods.lle_spectral import run_lle, run_spectral
from src.methods.umap_method  import run_umap
from src.methods.autoencoder  import (Autoencoder, encode_dataset, get_device)
from src.evaluation.metrics   import (compute_all_metrics, build_comparison_table)
from src.utils.plotting import savefig
from src.config import CFG

FIGURES = CFG['figures_dir']
MODELS  = CFG['models_dir']
METRICS = CFG['metrics_dir']
SEED    = CFG['random_seed']
%matplotlib inline
plt.rcParams.update({'figure.dpi': 100})

In [ ]:
# Cell 2 — Load datasets
DATA_DIR = CFG['data_dir']
file_map = {
    'Swiss Roll':   'swiss_roll.npz',
    'S-Curve':      's_curve.npz',
    'Torus':        'torus.npz',
    'Mobius Strip': 'mobius_strip.npz',
}

datasets = {}
for name, fname in file_map.items():
    path = DATA_DIR / fname
    if path.exists():
        d = np.load(path)
        datasets[name] = (d['X'], d['t'])
        print(f'  Loaded {name}: X={d["X"].shape}')
    else:
        print(f'  {fname} not found — regenerating')
        datasets = load_all_datasets(n_samples=CFG['n_samples'], noise=0.1, seed=SEED)
        break

In [ ]:
# Cell 3 — Load saved autoencoder checkpoints from Phase 6

def load_autoencoder(name: str, models_dir: Path) -> Autoencoder | None:
    fname = name.lower().replace(' ', '_') + '_ae.pt'
    path  = models_dir / fname
    if not path.exists():
        print(f'  Checkpoint not found: {fname} — will retrain')
        return None
    ckpt  = torch.load(path, map_location='cpu')
    model = Autoencoder(input_dim=3,
                         hidden_dims=ckpt['hidden_dims'],
                         latent_dim=ckpt['latent_dim'])
    model.load_state_dict(ckpt['model_state'])
    model.X_mean = ckpt['X_mean']
    model.X_std  = ckpt['X_std']
    model.eval()
    print(f'  Loaded checkpoint: {fname}')
    return model

ae_models = {}
for name in datasets:
    ae_models[name] = load_autoencoder(name, MODELS)

In [ ]:
# Cell 4 — Run all 7 methods on all 4 datasets
# This cell takes 3-5 minutes — all embeddings computed here

METHOD_CFG = {
    'PCA':          lambda X: run_pca(X),
    'Isomap':       lambda X: run_isomap(X, n_neighbors=10),
    'LLE':          lambda X: run_lle(X, method='standard',  n_neighbors=10),
    'LLE Modified': lambda X: run_lle(X, method='modified',  n_neighbors=20),
    'Spectral':     lambda X: run_spectral(X, n_neighbors=10),
    'UMAP':         lambda X: run_umap(X, n_neighbors=15, min_dist=0.1, random_state=SEED),
}

# Store: { dataset_name: { method_name: (X_emb, meta) } }
embeddings = {name: {} for name in datasets}

for ds_name, (X, t) in datasets.items():
    print(f'\n{ds_name}')
    for method_name, fn in METHOD_CFG.items():
        try:
            X_emb, meta = fn(X)
            embeddings[ds_name][method_name] = (X_emb, meta)
            t_s = meta.get('fit_time_s', 0) or 0
            print(f'  {method_name:<16} {t_s:.2f}s')
        except Exception as e:
            print(f'  {method_name:<16} FAILED: {e}')
            embeddings[ds_name][method_name] = (np.zeros((X.shape[0], 2)), {})

    # Autoencoder — from checkpoint
    model = ae_models.get(ds_name)
    if model is not None:
        Z = encode_dataset(model, X)
        embeddings[ds_name]['Autoencoder'] = (Z, {'fit_time_s': None})
        print(f'  {"Autoencoder":<16} (loaded from checkpoint)')
    else:
        print(f'  Autoencoder     checkpoint missing — run Phase 6 first')

In [ ]:
# Cell 5 — Compute all metrics
# trustworthiness + continuity + spearman for every (method, dataset) pair

print('Computing metrics (this takes 2-3 minutes)...')

metric_results = {}
for ds_name, (X, t) in datasets.items():
    metric_results[ds_name] = {}
    print(f'\n  {ds_name}')
    for method_name, (X_emb, meta) in embeddings[ds_name].items():
        # Autoencoder reconstruction for MSE
        X_hat = None
        if method_name == 'Autoencoder':
            model = ae_models.get(ds_name)
            if model is not None:
                X_norm = (X - model.X_mean) / model.X_std
                with torch.no_grad():
                    X_hat_norm = model(torch.FloatTensor(X_norm)).numpy()
                X_hat = X_hat_norm * model.X_std + model.X_mean

        m = compute_all_metrics(X, X_emb, k=10, X_hat=X_hat)
        m['fit_time_s'] = meta.get('fit_time_s')
        metric_results[ds_name][method_name] = m
        print(f'    {method_name:<16}  '
              f'TW={m["trustworthiness"]:.3f}  '
              f'Cont={m["continuity"]:.3f}  '
              f'Spear={m["spearman"]:.3f}')

In [ ]:
# Cell 6 — Build and save the master comparison table

df = build_comparison_table(metric_results)

print('='*70)
print('MASTER COMPARISON TABLE')
print('='*70)
print(df.to_string())

csv_path = METRICS / '06_master_comparison.csv'
df.to_csv(csv_path)
print(f'\nSaved -> results/metrics/06_master_comparison.csv')

In [ ]:
# Cell 7 — Heatmap: Trustworthiness across all methods x datasets

methods  = list(list(metric_results.values())[0].keys())
ds_names = list(metric_results.keys())

def metric_matrix(metric_key: str) -> np.ndarray:
    mat = np.full((len(ds_names), len(methods)), np.nan)
    for i, ds in enumerate(ds_names):
        for j, m in enumerate(methods):
            val = metric_results[ds].get(m, {}).get(metric_key, np.nan)
            mat[i, j] = val if val is not None else np.nan
    return mat

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Method comparison — all metrics (darker = better)', fontsize=13)

metric_labels = [
    ('trustworthiness', 'Trustworthiness', 'YlGn'),
    ('continuity',      'Continuity',      'YlGn'),
    ('spearman',        'Spearman r',      'YlGn'),
]

for ax, (key, title, cmap) in zip(axes, metric_labels):
    mat = metric_matrix(key)
    im  = ax.imshow(mat, cmap=cmap, vmin=0, vmax=1, aspect='auto')
    plt.colorbar(im, ax=ax)

    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels(methods, rotation=35, ha='right', fontsize=9)
    ax.set_yticks(range(len(ds_names)))
    ax.set_yticklabels(ds_names, fontsize=9)
    ax.set_title(title, fontsize=11)

    # Annotate cells
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                        fontsize=8, color='black' if val > 0.5 else 'gray')

plt.tight_layout()
savefig(fig, FIGURES / '06_heatmap_all_metrics.png')
plt.show()

In [ ]:
# Cell 8 — Radar / spider chart per dataset
# Shows each method's profile across all three metrics

from matplotlib.patches import FancyArrowPatch

metric_keys = ['trustworthiness', 'continuity', 'spearman']
metric_names = ['Trustworthiness', 'Continuity', 'Spearman r']
n_metrics    = len(metric_keys)
angles       = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
angles      += angles[:1]  # close the polygon

colors = ['#185FA5', '#1D9E75', '#BA7517', '#9B4FA0', '#E24B4A', '#378ADD', '#5C4033']

fig, axes = plt.subplots(2, 2, figsize=(14, 12),
                          subplot_kw=dict(polar=True))
fig.suptitle('Method profiles per dataset (radar chart)', fontsize=13)

for ax, ds_name in zip(axes.flat, ds_names):
    ax.set_title(ds_name, fontsize=11, pad=15)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metric_names, fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['0.25', '0.5', '0.75', '1.0'], fontsize=7)
    ax.grid(True, alpha=0.3)

    for (method, color) in zip(methods, colors):
        vals = []
        for k in metric_keys:
            v = metric_results[ds_name].get(method, {}).get(k, 0)
            vals.append(float(v) if v is not None and not np.isnan(float(v)) else 0)
        vals += vals[:1]
        ax.plot(angles, vals, 'o-', linewidth=1.5, color=color, markersize=3, label=method)
        ax.fill(angles, vals, alpha=0.05, color=color)

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
savefig(fig, FIGURES / '06_radar_per_dataset.png')
plt.show()

In [ ]:
# Cell 9 — Grand visual grid: all methods x all datasets
# The ultimate side-by-side comparison figure

all_methods  = list(embeddings[ds_names[0]].keys())
n_methods    = len(all_methods)
n_datasets   = len(ds_names)

fig, axes = plt.subplots(n_datasets, n_methods,
                          figsize=(n_methods * 3, n_datasets * 3))
fig.suptitle('All methods × all datasets', fontsize=15, y=1.01)

for row, ds_name in enumerate(ds_names):
    _, t = datasets[ds_name]
    for col, method in enumerate(all_methods):
        ax    = axes[row, col]
        X_emb = embeddings[ds_name][method][0]
        valid = np.std(X_emb) > 1e-6

        if valid:
            ax.scatter(X_emb[:, 0], X_emb[:, 1],
                       c=t, cmap='viridis', s=2, alpha=0.7)
        else:
            ax.text(0.5, 0.5, 'FAILED', transform=ax.transAxes,
                    ha='center', va='center', color='gray', fontsize=8)

        # TW score in corner
        tw = metric_results[ds_name].get(method, {}).get('trustworthiness')
        if tw and not np.isnan(tw):
            ax.text(0.02, 0.97, f'TW={tw:.2f}', transform=ax.transAxes,
                    fontsize=7, va='top', color='white',
                    bbox=dict(boxstyle='round,pad=0.2', fc='#333', alpha=0.6))

        if row == 0:
            ax.set_title(method, fontsize=9, fontweight='bold')
        if col == 0:
            ax.set_ylabel(ds_name, fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
savefig(fig, FIGURES / '06_grand_comparison_grid.png', dpi=200)
plt.show()
print('Saved: 06_grand_comparison_grid.png  ← main figure for your report')

In [ ]:
# Cell 10 — Speed comparison bar chart across all methods

speed_rows = []
for ds_name in ds_names:
    for method in all_methods:
        t_s = embeddings[ds_name][method][1].get('fit_time_s')
        if t_s:
            speed_rows.append({'Dataset': ds_name, 'Method': method, 'Time (s)': t_s})

df_speed = pd.DataFrame(speed_rows)

fig, ax = plt.subplots(figsize=(13, 5))
pivot   = df_speed.pivot(index='Method', columns='Dataset', values='Time (s)')
pivot.plot(kind='bar', ax=ax, colormap='Set2', alpha=0.85)
ax.set_ylabel('Fit time (seconds)', fontsize=11)
ax.set_title('Method speed comparison across datasets', fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.legend(title='Dataset', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
savefig(fig, FIGURES / '06_speed_comparison.png')
plt.show()

In [ ]:
# Cell 11 — Winner per dataset: best method by trustworthiness

print('Best method per dataset (by Trustworthiness):')
print('='*50)
for ds_name in ds_names:
    scores = {m: metric_results[ds_name][m]['trustworthiness']
              for m in metric_results[ds_name]
              if not np.isnan(metric_results[ds_name][m]['trustworthiness'])}
    if scores:
        best   = max(scores, key=scores.get)
        print(f'  {ds_name:<16}  {best:<16}  TW={scores[best]:.4f}')

print()
print('Best method per dataset (by Spearman r):')
print('='*50)
for ds_name in ds_names:
    scores = {m: metric_results[ds_name][m]['spearman']
              for m in metric_results[ds_name]
              if not np.isnan(metric_results[ds_name][m]['spearman'])}
    if scores:
        best = max(scores, key=scores.get)
        print(f'  {ds_name:<16}  {best:<16}  r={scores[best]:.4f}')

## Phase 7 Summary

| Figure | Description |
|---|---|
| `06_heatmap_all_metrics.png` | 3-panel heatmap: TW / Continuity / Spearman across all methods |
| `06_radar_per_dataset.png` | Spider charts showing each method's profile per dataset |
| `06_grand_comparison_grid.png` | Full visual grid — the main report figure |
| `06_speed_comparison.png` | Grouped bar chart of fit times |
| `06_master_comparison.csv` | Full numeric table — import into your report |

**Key takeaways to write about:**
- No single method wins on all datasets — the right choice depends on dataset topology
- Isomap + UMAP dominate on Swiss Roll (global structure methods)
- Spectral Embedding is the most consistent across all four datasets
- The Autoencoder is the only method that generalises to new points (parametric)
- The Torus and Möbius Strip expose fundamental topological limits of flat embeddings

**Next:** `Phase 8` — Streamlit demo + final report.